# sam4xtal on JupyterHub

This is the **supported** way to use sam4xtal inside JupyterHub.

| Piece | Role on JupyterHub |
| --- | --- |
| **Sidecar** (port 9001) | Real work — start once per session |
| **This notebook** | Front-end (click → segment → save) |
| **Next.js UI** | Optional / usually unreachable (Hub does not proxy arbitrary ports) |

### Once per session (JupyterLab terminal)

```bash
cd /path/to/sam4xtal          # prefer $SCRATCH or $WORK, not $HOME
chmod +x jupyterhub/*.sh
./jupyterhub/start_sidecar.sh          # GPU + HF weights
# ./jupyterhub/start_sidecar.sh --mock  # no GPU / smoke test
```

```bash
export HF_HOME=$SCRATCH/sam4xtal/hf-cache   # already default if $SCRATCH exists
export HF_TOKEN=…                            # gated facebook/sam3
```

**This kernel** needs: `ipywidgets`, `matplotlib`, `numpy`, `Pillow`, and **`ipympl`** for clickable figures.  
The sidecar installs into `sidecar/.venv` separately — do not expect this kernel to import `torch`.

## 1. Imports + sidecar

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from IPython.display import display

HERE = Path.cwd().resolve()
ROOT = None
for candidate in [HERE, HERE.parent, *HERE.parents]:
    if (candidate / "jupyterhub" / "sam4xtal_hub" / "workspace.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError(
        "Cannot find sam4xtal repo root (expected jupyterhub/sam4xtal_hub/). "
        "cd into the repo or notebooks/ and re-run."
    )

hub_path = str(ROOT / "jupyterhub")
if hub_path not in sys.path:
    sys.path.insert(0, hub_path)

print(f"ROOT = {ROOT}")

from sam4xtal_hub import ensure_sidecar, launch_ui

# True = flood-fill stub (no GPU / no HF weights)
USE_MOCK = False

client = ensure_sidecar(
    start_if_missing=True,
    mock=USE_MOCK,
    wait_s=900,
)
print(client.health())

## 2. Workspace UI

1. **Load** an image (samples: `samples/crystals/fig02_A.png`, …).
2. Click for **positive** / **negative** points (`ipympl` required).
3. **Segment active** → overlay.
4. **New instance** for another crystal on the same image.
5. **Save annotation** → `<image>.mask.json` + `<image>.mask.png` (same format as Next.js; use with `mask_statistics.ipynb`).

In [ ]:
# If figure clicks do nothing:
#   %pip install ipympl ipywidgets matplotlib Pillow
# then restart the kernel and re-run from cell 1.

%matplotlib widget

ui, state, client = launch_ui(client)
display(ui)

## 3. Optional: headless API (no widgets)

If `ipympl` is missing, set points by coordinates.

In [ ]:
from sam4xtal_hub.workspace import (
    WorkspaceState,
    load_image,
    segment_active,
    export_annotation,
    composite_overlay,
)
import matplotlib.pyplot as plt

# Example — uncomment and edit coordinates:
# st = WorkspaceState()
# load_image(st, ROOT / "samples/crystals/fig02_A.png")
# st.ensure_instance().points = [
#     {"x": 200.0, "y": 180.0, "positive": True},
# ]
# segment_active(client, st)
# print(st.instances[0].measurement())
# j, p = export_annotation(st)
# print("wrote", j, p)
# plt.figure(figsize=(6, 6))
# plt.imshow(composite_overlay(st.image_rgb, st.instances))
# plt.axis("off")
# plt.show()

## 4. Next.js? (usually skip)

```bash
./jupyterhub/start_web_optional.sh
```

- If the Hub has **jupyter-server-proxy**: try `/user/<you>/proxy/3000/`
- Or SSH tunnel (VPN): `ssh -L 3000:localhost:3000 you@<node>` → http://localhost:3000

This faculty Hub does not forward TensorBoard ports; expect the same for Next.js. Prefer this notebook.

## 5. Shutdown

```bash
./jupyterhub/stop_sidecar.sh
```

Also **stop the Hub server** when finished so the allocation (and GPU) is released.